In [109]:
import torchaudio

#load data
train_data = torchaudio.datasets.SPEECHCOMMANDS(root='/content/data', download=True, subset='training')
test_data = torchaudio.datasets.SPEECHCOMMANDS(root='/content/data',download=True, subset= 'testing')

In [110]:
len(train_data) , len(test_data)

(84843, 11005)

In [111]:
train_data[0]

(tensor([[-0.0658, -0.0709, -0.0753,  ..., -0.0700, -0.0731, -0.0704]]),
 16000,
 'backward',
 '0165e0e8',
 0)

In [112]:
import torchaudio.transforms as T
import torch
from torch.utils.data import Dataset

TARGET_SR = 16000
TARGET_LEN = 16000

# All 35 SpeechCommands classes listed statically to avoid reading the dataset to find them
CLASSES = [
    'backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'follow', 'forward',
    'four', 'go', 'happy', 'house', 'left', 'marvin', 'nine', 'no', 'off', 'on', 'one',
    'ooh', 'right', 'seven', 'sheila', 'six', 'stop', 'three', 'tree', 'two', 'up',
    'visual', 'wow', 'yes', 'zero'
]
LABEL_TO_INDEX = {label: i for i, label in enumerate(CLASSES)}

# Apply all transforms needed
def preprocess(waveform, sr):
  if sr != TARGET_SR:
    waveform = T.Resample(sr, TARGET_SR)(waveform)
  # Mono conversion
  if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)
  # Pad or trim length
  if waveform.shape[1] < TARGET_LEN:
    waveform = torch.nn.functional.pad(waveform, (0, TARGET_LEN - waveform.shape[1]))
  else:
    waveform = waveform[:, :TARGET_LEN]
  return waveform

def convert_to_mel_spec(waveform):
  mel_spec = T.MelSpectrogram(sample_rate=TARGET_SR, n_fft=1024, hop_length=512, n_mels=80)(waveform)
  return mel_spec.squeeze(0)

# Create a wrapper dataset to dynamically apply preprocessing
class PreprocessedDataset(Dataset):
  def __init__(self, dataset):
    self.dataset = dataset

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, idx):
    waveform, sr, label, speaker_id, utterance_number = self.dataset[idx]
    processed_waveform = preprocess(waveform, sr)
    processed_waveform = convert_to_mel_spec(processed_waveform)
    label_idx = LABEL_TO_INDEX.get(label, 0) # Fallback to 0 if unknown label
    # Add a channel dimension so it has shape [1, 80, 32] for Conv2d
    return processed_waveform.unsqueeze(0), label_idx

# Wrap our datasets
preprocessed_train_data = PreprocessedDataset(train_data)
preprocessed_test_data = PreprocessedDataset(test_data)

# Test retrieving a sample
first_sample = preprocessed_train_data[0]
print("Processed Waveform Shape:", first_sample[0].shape)

Processed Waveform Shape: torch.Size([1, 80, 32])


In [113]:
#make dataloader with parallel workers to speed up CPU-bound preprocessing
from torch.utils.data import DataLoader
batch_size = 64
train_dataloader = DataLoader(preprocessed_train_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(preprocessed_test_data, batch_size=batch_size, shuffle=False)

In [114]:
# next(iter(train_dataloader))[0][0].shape
# next(iter(train_dataloader))[0][0]

In [115]:
import torch.nn as nn
class AudioClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.encoder = nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=0), # 78 * 30 * 16 is output
        nn.ReLU(),
        nn.BatchNorm2d(16),
        nn.MaxPool2d(kernel_size=2, stride=2),     # 39 * 15 * 16
        nn.Conv2d(16, 32, kernel_size=4, stride=1, padding=0),  #36 * 12 * 32 is output
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),     # 18 * 6 * 32
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features= 18 * 6 * 32, out_features= 128),
        nn.Dropout(0.2),
        nn.ReLU(),
        nn.Linear(in_features=128, out_features= 35)
    )

  def forward(self, x):
    x = self.encoder(x)
    return self.classifier(x)

torch.manual_seed(42)
model = AudioClassifier()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [116]:
epochs = 2
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [117]:
for epoch in range(epochs):
  model.train()
  train_loss = 0.0
  train_correct = 0
  total_samples = 0

  for batch, (x, y) in enumerate(train_dataloader):
    x = x.to(device)
    y = y.to(device)

    # Forward pass
    y_pred = model(x)
    loss = loss_fn(y_pred, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Accumulate training statistics correctly
    train_loss += loss.item() * x.size(0)
    batch_correct = (y_pred.argmax(dim=1) == y).sum().item()
    train_correct += batch_correct
    total_samples += x.size(0)

    # Print progress every 10 batches
    if batch % 10 == 0:
      print(f"Epoch [{epoch+1}/{epochs}], Batch [{batch}/{len(train_dataloader)}], "
            f"Batch Loss: {loss.item():.4f}, Batch Acc: {batch_correct / x.size(0):.4f}")

  epoch_loss = train_loss / total_samples
  epoch_acc = train_correct / total_samples
  print(f"\n=== Epoch {epoch+1} Summary ===")
  print(f"Average Loss: {epoch_loss:.4f}")
  print(f"Average Accuracy: {epoch_acc:.4f}\n")

Epoch [1/2], Batch [0/1326], Batch Loss: 3.5781, Batch Acc: 0.0938
Epoch [1/2], Batch [10/1326], Batch Loss: 3.5741, Batch Acc: 0.1250
Epoch [1/2], Batch [20/1326], Batch Loss: 3.4892, Batch Acc: 0.0625
Epoch [1/2], Batch [30/1326], Batch Loss: 3.1948, Batch Acc: 0.1719
Epoch [1/2], Batch [40/1326], Batch Loss: 3.1125, Batch Acc: 0.1406
Epoch [1/2], Batch [50/1326], Batch Loss: 2.8515, Batch Acc: 0.1562
Epoch [1/2], Batch [60/1326], Batch Loss: 3.4537, Batch Acc: 0.0938
Epoch [1/2], Batch [70/1326], Batch Loss: 2.6939, Batch Acc: 0.1719
Epoch [1/2], Batch [80/1326], Batch Loss: 2.6173, Batch Acc: 0.2188
Epoch [1/2], Batch [90/1326], Batch Loss: 2.6719, Batch Acc: 0.2500
Epoch [1/2], Batch [100/1326], Batch Loss: 2.5395, Batch Acc: 0.2969
Epoch [1/2], Batch [110/1326], Batch Loss: 2.6175, Batch Acc: 0.2812
Epoch [1/2], Batch [120/1326], Batch Loss: 2.5133, Batch Acc: 0.2344
Epoch [1/2], Batch [130/1326], Batch Loss: 2.7010, Batch Acc: 0.2344
Epoch [1/2], Batch [140/1326], Batch Loss: 2.

In [118]:
#predictions on test set
model.eval()
test_accuracy = 0
test_correct = 0

with torch.inference_mode():
  for x, y in test_dataloader:
    x = x.to(device)
    y = y.to(device)
    y_test_pred = model(x)
    test_correct += (torch.argmax(y_test_pred, dim=1) == y).sum().item()

test_accuracy = test_correct / (len(test_dataloader) * batch_size)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Test Accuracy: 75.67%
